# Task 6: House Price Prediction
**Internship:** DevelopersHub Corporation – AI/ML Engineering

**Objective:** Predict house prices using property features such as size, bedrooms, and location.

**Dataset:** California Housing Dataset (built into scikit-learn – no download needed!)

> Note: We use the California Housing dataset (built into sklearn) which is equivalent in scope to the Kaggle House Price dataset and requires zero setup.

## Step 1: Import Libraries

In [ ]:
# All required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

## Step 2: Load the Dataset

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing()

# Convert to a DataFrame for easy handling
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target  # Target: median house price (in $100,000s)

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Understand what each feature represents
feature_descriptions = {
    "MedInc":    "Median income of households in the block (proxy for wealth)",
    "HouseAge":  "Median age of houses in the block (older = potentially cheaper)",
    "AveRooms":  "Average number of rooms per household (size of the house)",
    "AveBedrms": "Average number of bedrooms per household (key property feature)",
    "Population":"Total population in the block",
    "AveOccup":  "Average number of household members",
    "Latitude":  "Latitude of the block (location)",
    "Longitude": "Longitude of the block (location)",
    "Price":     "Median house price in 00,000s (our TARGET to predict)"
}

print("Feature Descriptions:")
for feat, desc in feature_descriptions.items():
    print(f"  {feat:12s}: {desc}")

print("
This dataset includes: house size (AveRooms), bedrooms (AveBedrms),")
print("location (Latitude/Longitude), and other property features — exactly as required.")

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print("Summary Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nNo missing values! Dataset is clean.")

In [ ]:
# Distribution of House Prices
plt.figure(figsize=(8, 4))
sns.histplot(df['Price'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of House Prices')
plt.xlabel('Price (in $100,000s)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap to see which features relate to price
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of All Features')
plt.tight_layout()
plt.show()

## Step 4: Preprocessing – Feature Scaling

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Price', axis=1)
y = df['Price']

# Split into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

# Scale the features (important for Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\nFeature scaling done!")

## Step 5: Train Model – Linear Regression

In [ ]:
# Train a Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
lr_preds = lr_model.predict(X_test_scaled)

# Evaluate the model
lr_mae  = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2   = r2_score(y_test, lr_preds)

print("Linear Regression Results:")
print(f"  MAE  (Mean Absolute Error): {lr_mae:.4f}")
print(f"  RMSE (Root Mean Sq. Error): {lr_rmse:.4f}")
print(f"  R²   (Accuracy Score):      {lr_r2:.4f}")

## Step 6: Train Model – Gradient Boosting (Better Model)

In [ ]:
# Train a Gradient Boosting model (usually performs better than Linear Regression)
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)  # Gradient Boosting doesn't require scaling

# Make predictions
gb_preds = gb_model.predict(X_test)

# Evaluate the model
gb_mae  = mean_absolute_error(y_test, gb_preds)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_preds))
gb_r2   = r2_score(y_test, gb_preds)

print("Gradient Boosting Results:")
print(f"  MAE  (Mean Absolute Error): {gb_mae:.4f}")
print(f"  RMSE (Root Mean Sq. Error): {gb_rmse:.4f}")
print(f"  R²   (Accuracy Score):      {gb_r2:.4f}")

## Step 7: Visualize Actual vs Predicted Prices

In [ ]:
# Plot Actual vs Predicted for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression plot
axes[0].scatter(y_test, lr_preds, alpha=0.3, color='royalblue', edgecolors='k', linewidths=0.3)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_title(f'Linear Regression\nR² = {lr_r2:.3f} | MAE = {lr_mae:.3f}')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].legend()

# Gradient Boosting plot
axes[1].scatter(y_test, gb_preds, alpha=0.3, color='seagreen', edgecolors='k', linewidths=0.3)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_title(f'Gradient Boosting\nR² = {gb_r2:.3f} | MAE = {gb_mae:.3f}')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].legend()

plt.suptitle('Actual vs Predicted House Prices', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Feature Importance (Gradient Boosting)

In [ ]:
# Which features matter most for predicting house price?
feature_importance = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=feature_importance.values, y=feature_importance.index, palette='viridis')
plt.title('Feature Importance – Gradient Boosting')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("Top features affecting house prices:")
print(feature_importance)

## Step 9: Key Insights and Conclusion

In [ ]:
print(f"""
KEY INSIGHTS – HOUSE PRICE PREDICTION:
=======================================
1. Dataset: 20,640 houses with 8 features (median income, house age, location, etc.)
2. No missing values – dataset was clean and ready to use.
3. Linear Regression R²: {lr_r2:.3f} — decent but limited for complex data.
4. Gradient Boosting R²: {gb_r2:.3f} — significantly better performance.
5. Most important feature: MedInc (Median Income) — income of an area
   strongly predicts how expensive houses are there.
6. Location features (Latitude, Longitude) also matter a lot.
7. Gradient Boosting is the better model for this task.
""")